# Generatív MI API-k és Integrációk – 7. Hét

**Téma:** Szövegalapú modell API-k

**Használt modellek:** 
- **Groq** (cloud, ingyenes free tier):
  - `llama-3.1-8b-instant` – gyors, alacsony késleltetésű modell
  - `llama-3.3-70b-versatile` – erősebb, kiváló logikai következtetés és strukturált kimenet
- **Hugging Face** – lokális embeddingek (teljesen offline és ingyenes)

**Előkészületek:**
1. Regisztrálj a https://console.groq.com oldalon → másold ki az API kulcsot (ingyenes, hitelkártya nem szükséges)
2. Hozz létre egy `.env` fájlt a notebook könyvtárában az alábbi tartalommal:
   ```env
   GROQ_API_KEY=gq_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
   ```

Limitek: https://console.groq.com/docs/rate-limits 

In [1]:
# Importok – minden szükséges könyvtár egy helyen
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from pydantic import BaseModel, Field
from typing import List
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

load_dotenv()  # Betölti a .env fájlból az API kulcsot

print("Környezet sikeresen betöltve")

/Applications/anaconda3/envs/genai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Környezet sikeresen betöltve


## 1. Egyszerű kérdés-válasz Groq modellekkel

Először nézzük meg, hogyan lehet egy egyszerű kérdést elküldeni a Groq modelljeinek.

In [2]:
def groq_simple_query(query, model_name="llama-3.3-70b-versatile"):
    """Elküld egy kérdést a megadott Groq modellnek és visszaadja a választ szövegként."""
    try:
        llm = ChatGroq(model=model_name)

        response = llm.invoke(query)

        return response.content.strip()
    except Exception as e:
        return f"Hiba történt a hívás során: {str(e)}"

# Teszteljük a két aktuális Groq modellt
models_to_test = [
    "llama-3.1-8b-instant",       # Gyors, alacsony késleltetésű modell
    "llama-3.3-70b-versatile"     # Erősebb, jobb minőségű válaszok
]

query = "Mi Magyarország fővárosa?"

print(f"Kérdés: {query}\n")

for model in models_to_test:
    print(f"{model}:")

    print(groq_simple_query(query, model))
    
    print("─" * 50)

Kérdés: Mi Magyarország fővárosa?

llama-3.1-8b-instant:
Magyarország fővárosa Budapest.
──────────────────────────────────────────────────
llama-3.3-70b-versatile:
Budapest Magyarország fővárosa.
──────────────────────────────────────────────────


## 2. Chat history kezelése (kontextus megőrzése)

Most készítünk egy interaktív chatet, amely emlékszik a korábbi üzenetekre.

In [3]:
def chat_with_groq():
    """Interaktív chat a Groq erősebb modelljével – a teljes beszélgetés history-ja megmarad."""

    model = ChatGroq(model="llama-3.3-70b-versatile")

    chat_history = [
        SystemMessage(content="Te egy segítőkész, magyarul válaszoló asszisztens vagy.")
    ]
    
    print("Groq Chat elindítva! Írd be, hogy 'exit' vagy 'kilépés' a befejezéshez.\n")
    
    while True:
        user_input = input("Te: ")

        if user_input.lower() in ["exit", "kilépés"]:
            break
            
        chat_history.append(HumanMessage(content=user_input))

        response = model.invoke(chat_history)

        print(f"Bot: {response.content}\n")
        
        chat_history.append(AIMessage(content=response.content))
        

chat_with_groq()  # Futtasd ezt a sort, ha ki szeretnéd próbálni a chatet

Groq Chat elindítva! Írd be, hogy 'exit' vagy 'kilépés' a befejezéshez.

Bot: Üdvözöllek! Örülök, hogy itt vagy. Hogyan segíthetek neked ma? Van egy kérdésed, vagy csak szeretnél beszélgetni?

Bot: Nem történt semmi, mert nem írtál semmit. Kérlek, írj valamit, és én válaszolok. Beszélgessünk!



## 3. Prompt template használata

A prompt template-ek segítségével egységes stílusban kérdezhetünk különböző témákról.

In [4]:
template = ChatPromptTemplate.from_messages([
    ("system", "Te egy szakértő vagy a(z) {domain} területén. Magyarul válaszolj."),
    ("human", "Magyarázd el egy mondatban: {topic}")
])

def explain_with_groq(domain: str, topic: str):
    """Egy adott témát szakértői szerepben magyaráz el egy mondatban."""

    model = ChatGroq(model="llama-3.3-70b-versatile")

    chain = template | model

    response = chain.invoke({"domain": domain, "topic": topic})

    return response.content

print(explain_with_groq("mesterséges intelligencia", "neurális hálók"))

print(explain_with_groq("történelem", "második világháború"))

print(explain_with_groq("fizika", "kvantummechanika"))

A neurális hálók olyan számítógépes modellek, amelyek az emberi agy működését utánozva, sok rétegből és csomópontból álló hálózatokat használnak feladatok, mint például a képfelismerés, a beszédfelismerés és a gépi tanulás megoldására.
A második világháború egy globális konfliktus volt, amely 1939 és 1945 között zajlott, és a tengelyhatalmak (Németország, Olaszország, Japán) és a szövetségesek (Egyesült Államok, Egyesült Királyság, Szovjetunió) közötti összecsapások során az emberiség történetének egyik legsúlyosabb és legpusztítóbb háborújává vált.
A kvantummechanika a fizika azon ága, amely az anyag és energia viselkedését tanulmányozza az atomi és szubatomi szinten, ahol a klasszikus fizika törvényei már nem érvényesek, és a jelenségek a valószínűség és a hullámfüggvények segítségével írhatók le.


## 4. Embeddingek és hasonlósági keresés (Hugging Face – teljesen lokális)

Szövegeket vektorrá alakítunk, majd keresünk a legjobban illeszkedő dokumentumokra.

In [5]:
def find_most_similar(query: str, docs: List[str], top_k: int = 3):
    """Visszaadja a query-hez legjobban illeszkedő top_k dokumentumot hasonlósági pontszámmal."""

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    doc_emb = embeddings.embed_documents(docs)
    query_emb = embeddings.embed_query(query)
    
    scores = cosine_similarity([query_emb], doc_emb)[0]

    top_indices = np.argsort(scores)[::-1][:top_k]
    
    return [(docs[i], round(scores[i], 4)) for i in top_indices]

# Példa dokumentumok magyar focistákról
docs = [
    "Lionel Messi az Inter Miami játékosa, argentin válogatott. Őt tartják minden idők egyik legjobb labdarúgójának.",
    "Cristiano Ronaldo az Al Nassr csatára, portugál válogatott. Híres atletikusságáról és gólérzékenységéről.",
    "Kylian Mbappé a Paris Saint-Germain játékosa, francia válogatott. Hihetetlen sebességével tűnik ki.",
    "Erling Haaland a Manchester City csatára, norvég. Gólrekordere van több ligában is.",
    "Kevin De Bruyne a Manchester City középpályása, belga. A világ egyik legjobb passzolójának tartják."
]

query = "Ki játszik a Paris Saint-German csapatban?"

results = find_most_similar(query, docs, top_k=3)

print(f"Keresés: '{query}'\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"{i}. [hasonlóság: {score}] {doc}\n")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9981.59it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Keresés: 'Ki játszik a Paris Saint-German csapatban?'

1. [hasonlóság: 0.4708] Kylian Mbappé a Paris Saint-Germain játékosa, francia válogatott. Hihetetlen sebességével tűnik ki.

2. [hasonlóság: 0.4483] Erling Haaland a Manchester City csatára, norvég. Gólrekordere van több ligában is.

3. [hasonlóság: 0.392] Kevin De Bruyne a Manchester City középpályása, belga. A világ egyik legjobb passzolójának tartják.



## 5. Strukturált kimenet (JSON/Pydantic formátum)

A modell pontos, előre definiált struktúrában adja vissza az eredményt (pl. review elemzés).

In [6]:
class ReviewAnalysis(BaseModel):
    """Egy termékreview elemzésének előre definiált struktúrája."""
    
    key_themes: list[str] = Field(description="A review fő témái lista formában")
    summary: str = Field(description="Rövid, egy-két mondatos összefoglaló")
    
    # ["pozitív", "negatív", "semleges"]
    sentiment: str = Field(description="A review általános hangulata")
    
    pros: list[str] | None = Field(default=None, description="A review-ban említett előnyök listája")
    cons: list[str] | None = Field(default=None, description="A review-ban említett hátrányok listája")
    
    reviewer_name: str | None = Field(default=None, description="Az értékelő neve, ha meg van adva")


def analyze_review_groq(text):
    """Elemzi a megadott review szöveget és strukturált objektumot ad vissza."""
    model = ChatGroq(model="llama-3.3-70b-versatile")

    structured = model.with_structured_output(ReviewAnalysis)
    
    return structured.invoke(text)

# Példa egy magyar nyelvű review-val
review_text = """
Nemrég vettem egy Samsung Galaxy S24 Ultrát, és nagyon elégedett vagyok vele! A processzor villámgyors, a kamera fantasztikus, főleg éjszaka. Az akku egész nap bírja, az S-Pen pedig szuper jegyzeteléshez.
Ami nem tetszik: nagyon nehéz, egykezes használat szinte lehetetlen, tele van felesleges Samsung appokkal, és az ár is borsos.
Értékelés: Nitish Singh
"""

result = analyze_review_groq(review_text)

print(result)

print("\nJSON formátumban:")

print(result.model_dump_json(indent=2))

key_themes=['teljesítmény', 'kamera', 'akku', 'S-Pen'] summary='A Samsung Galaxy S24 Ultra egy nagyszerű eszköz, de van néhány hátránya, mint a súlya és az ára.' sentiment='pozitív' pros=['villámgyors processzor', 'fantasztikus kamera', 'akku egész nap bírja', 'S-Pen szuper jegyzeteléshez'] cons=['nagyon nehéz', 'egykezes használat szinte lehetetlen', 'tele van felesleges Samsung appokkal', 'az ár is borsos'] reviewer_name='Nitish Singh'

JSON formátumban:
{
  "key_themes": [
    "teljesítmény",
    "kamera",
    "akku",
    "S-Pen"
  ],
  "summary": "A Samsung Galaxy S24 Ultra egy nagyszerű eszköz, de van néhány hátránya, mint a súlya és az ára.",
  "sentiment": "pozitív",
  "pros": [
    "villámgyors processzor",
    "fantasztikus kamera",
    "akku egész nap bírja",
    "S-Pen szuper jegyzeteléshez"
  ],
  "cons": [
    "nagyon nehéz",
    "egykezes használat szinte lehetetlen",
    "tele van felesleges Samsung appokkal",
    "az ár is borsos"
  ],
  "reviewer_name": "Nitish Singh"
